# Teks — Dokumen panjang: potong jadi chunk bertumpuk, fine-tune, gabungkan per dokumen (OOF)
**Kapan dipakai:** teks jauh lebih panjang dari 512 token (berita, artikel, putusan, laporan,
esai) dan info penting bisa ada di mana saja. Memotong di 128/512 token pertama membuang isi;
`HEAD_TAIL` di `finetune_oof` hanya mengambil awal + akhir. Di sini SELURUH dokumen dibaca.

**Cara:** tokenizer memecah tiap dokumen menjadi jendela `MAX_LENGTH` token yang bertumpuk
`STRIDE` token. Tiap chunk mewarisi label dokumennya, model dilatih di chunk, lalu probabilitas
chunk dirata-rata (alt: maksimum) per dokumen. Fold dibagi per DOKUMEN, jadi chunk dokumen yang
sama tidak pernah ada di latih dan validasi sekaligus (OOF tetap format bersama).

**Saat lomba yang diubah:** sel Konfigurasi. Waktu sebanding jumlah chunk: batasi dengan
`MAX_CHUNKS`. Waktu mepet: `FOLDS_TO_RUN = [0]`.

**Data demo:** arXiv (`arxiv`, Inggris, median ± 400 token, 2.000 dokumen).

In [ ]:
import html
import os
import time
from collections import Counter

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import StratifiedKFold
from torch.nn.functional import cross_entropy, softmax
from torch.nn.utils import clip_grad_norm_
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from transformers import get_linear_schedule_with_warmup

## Konfigurasi

In [ ]:
DATA_FOLDER = "../../../data_latihan/text/arxiv"              # GANTI saat lomba
TRAIN_CSV = f"{DATA_FOLDER}/arxiv_train.csv"                   # GANTI saat lomba
TEST_CSV = f"{DATA_FOLDER}/arxiv_test.csv"                     # GANTI saat lomba
OUTPUT_FOLDER = "../../../outputs/text/dokumen_panjang"
OOF_FOLDER = "../../../outputs/text/oof"
SAMPLE_CSV = f"{OUTPUT_FOLDER}/sample_submission_demo.csv"     # GANTI saat lomba: file panitia
SUBMISSION_NAME = "submission.csv"
ANSWER_CSV = f"{DATA_FOLDER}/arxiv_test_labels.csv"            # HAPUS saat lomba (kunci demo)
ID_COL = "id"                    # GANTI saat lomba
TEXT_COL = "text"                # GANTI saat lomba
TARGET = "label"                 # GANTI saat lomba
SUBMIT_COL = TARGET
N_FOLDS = 5
SPLIT_SEED = 42
NOTEBOOK = "dokumen_panjang"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_BF16 = DEVICE == "cuda"

In [ ]:
PRETRAINED = "../../../pretrained/distilbert_en"      # GANTI: demo Inggris
# PRETRAINED = "../../../pretrained/indobert"         # berita/dokumen Indonesia
# PRETRAINED = "../../../pretrained/xlmr_base"        # campur bahasa
MODEL_NAME = "chunk_distilbert"  # GANTI tiap variasi (nama file OOF)
MAX_LENGTH = 256                 # panjang tiap chunk (128 lebih cepat, 512 konteks lebih utuh)
STRIDE = 64                      # token tumpang tindih antar chunk berurutan
MAX_CHUNKS = 8                   # chunk per dokumen maksimum (sisanya dibuang). None = semua
AGGREGATE = "mean"               # rata-rata probabilitas chunk
# AGGREGATE = "max"              # satu bagian saja cukup menentukan label (mis. ada kata kunci)
BATCH_SIZE = 16
EPOCHS = 2
LEARNING_RATE = 3e-5
WARMUP = 0.1
FOLDS_TO_RUN = None              # None = semua fold. Waktu mepet: [0] (OOF tidak disimpan)
SEED = 42

## Metrik

In [ ]:
METRIC_NAME = "f1_macro"         # GANTI saat lomba, ikut baris return di bawah


def score(truth, predicted):
    return f1_score(truth, predicted, average="macro")


print("metrik:", METRIC_NAME, "| device:", DEVICE)

## Baca data
Teks kosong/NaN jadi string kosong, lalu `clean_basic` (html + spasi rapi). Label diubah ke indeks
0..K-1 urut `np.unique` (format OOF bersama).

In [ ]:
os.makedirs(OUTPUT_FOLDER, exist_ok=True)
os.makedirs(OOF_FOLDER, exist_ok=True)
train = pd.read_csv(TRAIN_CSV)
test = pd.read_csv(TEST_CSV)


def clean_basic(text):
    return " ".join(html.unescape(text).split())


train_text = train[TEXT_COL].fillna("").astype(str)
test_text = test[TEXT_COL].fillna("").astype(str)
train_text = train_text.apply(clean_basic)
test_text = test_text.apply(clean_basic)
classes = np.unique(train[TARGET])
y = np.searchsorted(classes, train[TARGET].to_numpy())
print("train:", train.shape, "| test:", test.shape, "| NaN target:", train[TARGET].isna().sum())
print("kelas:", list(classes), "| jumlah:", np.bincount(y).tolist())
train[[ID_COL, TEXT_COL, TARGET]].head()

## Buat sample_submission demo (HAPUS saat lomba)
Data latihan tidak punya `sample_submission`. Saat lomba file panitia dipakai (`SAMPLE_CSV`);
sel ini tidak menimpa file di luar `OUTPUT_FOLDER`.

In [ ]:
demo_sample = pd.DataFrame()
demo_sample[ID_COL] = test[ID_COL]
demo_sample[SUBMIT_COL] = train[TARGET].iloc[0]
if SAMPLE_CSV.startswith(OUTPUT_FOLDER):
    demo_sample.to_csv(SAMPLE_CSV, index=False)
    print("sample demo ditulis ke", SAMPLE_CSV)
else:
    print("SAMPLE_CSV bukan file demo: dibiarkan apa adanya")

In [ ]:
sample = pd.read_csv(SAMPLE_CSV)
print("sample_submission:", sample.shape, sample.columns.tolist())
print("id test = id sample:", set(test[ID_COL]) == set(sample[ID_COL]))
sample.head(3)

## Split (k-fold)
Aturan format OOF: `StratifiedKFold` acak seed `SPLIT_SEED` pada urutan baris train apa adanya,
jadi OOF notebook ini sefold dengan `finetune_oof`, `baseline_oof`, dll. dan bisa di-ensemble.

In [ ]:
splitter = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SPLIT_SEED)
folds = list(splitter.split(train_text, y))
val_sizes = []
for _, val_index in folds:
    val_sizes.append(len(val_index))
print("fold:", len(folds), "| ukuran validasi:", val_sizes)

## Potong jadi chunk
`owner[i]` = nomor dokumen pemilik chunk ke-i. Chunk setelah `MAX_CHUNKS` per dokumen dibuang.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(PRETRAINED, local_files_only=True)


def make_chunks(texts):
    encoded = tokenizer(list(texts), truncation=True, max_length=MAX_LENGTH, stride=STRIDE,
                        return_overflowing_tokens=True)
    owner = np.array(encoded["overflow_to_sample_mapping"])
    keys = []
    for key in encoded.keys():
        if key != "overflow_to_sample_mapping":
            keys.append(key)
    rows, kept, seen = [], [], Counter()
    for number, doc in enumerate(owner):
        seen[doc] += 1
        if MAX_CHUNKS is None or seen[doc] <= MAX_CHUNKS:
            row = {}
            for key in keys:
                row[key] = encoded[key][number]
            rows.append(row)
            kept.append(doc)
    return rows, np.array(kept)


print("make_chunks siap | MAX_LENGTH", MAX_LENGTH, "STRIDE", STRIDE)

In [ ]:
train_rows, train_owner = make_chunks(train_text)
test_rows, test_owner = make_chunks(test_text)
per_doc = np.bincount(train_owner, minlength=len(train))
print("chunk train:", len(train_rows), "| test:", len(test_rows))
print("chunk per dokumen p50/p95/maks:", np.percentile(per_doc, [50, 95]), per_doc.max())
print("dokumen terpotong MAX_CHUNKS:", (per_doc >= (MAX_CHUNKS or 10**9)).sum())

## Gabung probabilitas chunk ke dokumen

In [ ]:
def to_documents(chunk_proba, owner, n_docs):
    if AGGREGATE == "max":
        doc_proba = np.zeros((n_docs, chunk_proba.shape[1]), dtype="float32")
        np.maximum.at(doc_proba, owner, chunk_proba)
        return doc_proba / np.maximum(doc_proba.sum(axis=1, keepdims=True), 1e-9)
    doc_proba = np.zeros((n_docs, chunk_proba.shape[1]), dtype="float32")
    np.add.at(doc_proba, owner, chunk_proba)
    counts = np.maximum(np.bincount(owner, minlength=n_docs), 1)
    return doc_proba / counts[:, None]


print("agregasi:", AGGREGATE)

## Model, latih, prediksi
Fine-tune penuh biasa (AMP bf16, warmup linear, clip 1.0) di level chunk.

In [ ]:
def pick(rows, chosen):
    picked = []
    for number in chosen:
        picked.append(rows[number])
    return picked


def forward_logits(model, rows):
    batch = tokenizer.pad(rows, return_tensors="pt").to(DEVICE)
    with torch.autocast("cuda", dtype=torch.bfloat16, enabled=USE_BF16):
        return model(**batch).logits.float()


def predict_chunks(model, rows):
    model.eval()
    proba = np.zeros((len(rows), len(classes)), dtype="float32")
    with torch.no_grad():
        for start in range(0, len(rows), BATCH_SIZE * 4):
            chosen = np.arange(start, min(start + BATCH_SIZE * 4, len(rows)))
            proba[chosen] = softmax(forward_logits(model, pick(rows, chosen)), dim=1).cpu().numpy()
    return proba


print("predict_chunks siap")

In [ ]:
def build_run(n_chunks):
    torch.manual_seed(SEED)
    model = AutoModelForSequenceClassification.from_pretrained(
        PRETRAINED, num_labels=len(classes), ignore_mismatched_sizes=True, local_files_only=True,
        dtype=torch.float32)
    model = model.to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
    steps = max(1, int(np.ceil(n_chunks / BATCH_SIZE))) * EPOCHS
    scheduler = get_linear_schedule_with_warmup(optimizer, int(WARMUP * steps), steps)
    return {"model": model, "optimizer": optimizer, "scheduler": scheduler,
            "rng": np.random.default_rng(SEED)}


def train_epoch(run, fit_chunks):
    model = run["model"]
    model.train()
    order = run["rng"].permutation(fit_chunks)
    for start in range(0, len(order), BATCH_SIZE):
        chosen = order[start:start + BATCH_SIZE]
        targets = torch.tensor(y[train_owner[chosen]]).to(DEVICE)
        loss = cross_entropy(forward_logits(model, pick(train_rows, chosen)), targets)
        loss.backward()
        clip_grad_norm_(model.parameters(), 1.0)
        run["optimizer"].step()
        run["scheduler"].step()
        run["optimizer"].zero_grad()


print("build_run / train_epoch siap")

In [ ]:
def train_fold(fit_index, val_index):
    fit_chunks = np.where(np.isin(train_owner, fit_index))[0]
    val_chunks = np.where(np.isin(train_owner, val_index))[0]
    run = build_run(len(fit_chunks))
    for epoch in range(EPOCHS):
        began = time.time()
        train_epoch(run, fit_chunks)
        chunk_proba = predict_chunks(run["model"], pick(train_rows, val_chunks))
        doc_proba = to_documents(chunk_proba, train_owner[val_chunks], len(train))[val_index]
        value = score(y[val_index], doc_proba.argmax(axis=1))
        print(f"  epoch {epoch + 1}: {METRIC_NAME} dokumen {value:.4f} | "
              f"{time.time() - began:.0f} dtk")
    test_part = to_documents(predict_chunks(run["model"], test_rows), test_owner, len(test))
    del run
    torch.cuda.empty_cache()
    return doc_proba, test_part


print("train_fold siap (bobot epoch terakhir)")

## Latih k-fold

In [ ]:
folds_to_run = FOLDS_TO_RUN
if folds_to_run is None:
    folds_to_run = list(range(len(folds)))
oof = np.zeros((len(train), len(classes)), dtype="float32")
test_proba = np.zeros((len(test), len(classes)), dtype="float32")
began = time.time()
for position, number in enumerate(folds_to_run):
    fit_index, val_index = folds[number]
    val_proba, test_part = train_fold(fit_index, val_index)
    oof[val_index] = val_proba
    test_proba += test_part / len(folds_to_run)
    left = (time.time() - began) / (position + 1) * (len(folds_to_run) - position - 1)
    print(f"fold {number} selesai | sisa ± {left / 60:.1f} menit")
total_seconds = round(time.time() - began, 1)
note = (f"{os.path.basename(PRETRAINED)} chunk {MAX_LENGTH}/{STRIDE} maks {MAX_CHUNKS} "
        f"{AGGREGATE} | epoch {EPOCHS} lr {LEARNING_RATE} | fold {folds_to_run}")
print(note, f"| total {total_seconds / 60:.1f} menit")

## Evaluasi OOF
Skor dihitung di baris train yang punya prediksi OOF (semua baris bila semua fold jalan).

In [ ]:
filled = oof.sum(axis=1) > 0
oof_pred = oof.argmax(axis=1)
oof_score = score(y[filled], oof_pred[filled])
print(f"{METRIC_NAME} OOF ({filled.sum()} baris): {oof_score:.4f}")
print(classification_report(y[filled], oof_pred[filled], target_names=classes.astype(str)))

## Simpan OOF dan catat percobaan
`oof_<MODEL_NAME>.npy` / `test_<MODEL_NAME>.npy` hanya bila semua baris train terisi, untuk
`ensemble_oof`. `percobaan.csv` bertambah satu baris tiap run.

In [ ]:
if filled.all():
    np.save(f"{OOF_FOLDER}/oof_{MODEL_NAME}.npy", oof)
    np.save(f"{OOF_FOLDER}/test_{MODEL_NAME}.npy", test_proba)
    print("OOF dan test disimpan:", MODEL_NAME)
else:
    print("OOF tidak lengkap (FOLDS_TO_RUN?): tidak disimpan")
row = {"waktu": time.strftime("%Y-%m-%d %H:%M:%S"), "notebook": NOTEBOOK, "model": MODEL_NAME,
       "metrik": METRIC_NAME, "skor_cv": round(oof_score, 4), "detik": total_seconds,
       "catatan": note}
log_path = f"{OOF_FOLDER}/percobaan.csv"
pd.DataFrame([row]).to_csv(log_path, mode="a", header=not os.path.exists(log_path), index=False)
history = pd.read_csv(log_path)
print(history.tail(5).to_string())

## Submission
Kolom id diambil dari `sample_submission`; prediksi dipetakan lewat id.

In [ ]:
test_pred = test_proba.argmax(axis=1)
pred_by_id = pd.Series(classes[test_pred], index=test[ID_COL])

submission = pd.DataFrame()
submission[ID_COL] = sample[ID_COL]
submission[SUBMIT_COL] = sample[ID_COL].map(pred_by_id)
submission.to_csv(f"{OUTPUT_FOLDER}/{SUBMISSION_NAME}", index=False)
predicted_share = submission[SUBMIT_COL].value_counts(normalize=True)
print("sebaran prediksi test:", predicted_share.round(3).to_dict())
submission.head()

## Cek submission terhadap sample_submission
Salinan validator `arsenal/umum/simpan_muat` #15 dan #16: hanya membaca dan mencetak. Hasil
akhir harus `MASALAH: 0`.

In [ ]:
SUBMISSION_CSV = f"{OUTPUT_FOLDER}/{SUBMISSION_NAME}"
ID_COLUMN = ID_COL

sample = pd.read_csv(SAMPLE_CSV)
written = pd.read_csv(SUBMISSION_CSV)
problems = []


def report(failed, text):
    if failed:
        problems.append(text)


wanted = sample[ID_COLUMN]
ids = written.get(ID_COLUMN, pd.Series(dtype=wanted.dtype))
lost = set(wanted) - set(ids)
extra = set(ids) - set(wanted)
kept = ids[ids.isin(wanted)].drop_duplicates().tolist()
expected = wanted[wanted.isin(ids)].tolist()
report(written.columns.tolist() != sample.columns.tolist(), "kolom / urutan kolom beda")
report(len(written) != len(sample), f"jumlah baris {len(written)}, sample {len(sample)}")
report(lost, f"id hilang {len(lost)}, mis. {sorted(lost)[:3]}")
report(extra, f"id asing {len(extra)}, mis. {sorted(extra)[:3]}")
report(ids.duplicated().any(), f"id kembar: {ids.duplicated().sum()} baris")
report(kept != expected, "urutan id beda dari sample")

In [ ]:
LABEL_COLUMNS = [SUBMIT_COL]     # kolom isi kelas ([] = lewati cek label)
ALLOWED = list(classes)          # label sah = label train, [] = lewati cek

for column in sample.columns:
    if column in written.columns:
        empty = written[column].isna().sum()
        report(empty, f"{column}: {empty} nilai kosong")
        report(written[column].dtype != sample[column].dtype,
               f"{column}: tipe {written[column].dtype}, sample {sample[column].dtype}")
for column in LABEL_COLUMNS:
    if ALLOWED and column in written.columns:
        foreign = set(written[column].dropna()) - set(ALLOWED)
        report(foreign, f"{column}: label di luar ALLOWED {sorted(foreign)[:5]}")
        print(written[column].value_counts())   # semua satu kelas -> curigai model
for problem in problems:
    print("-", problem)
print("MASALAH:", len(problems))

## Cek dengan kunci jawaban (HAPUS saat lomba)
Hanya untuk data latihan yang punya kunci (`ANSWER_CSV`). Hapus sel ini saat lomba.

In [ ]:
answers = pd.read_csv(ANSWER_CSV)
truth_by_id = pd.Series(answers[TARGET].to_numpy(), index=answers[ID_COL])
truth_index = np.searchsorted(classes, truth_by_id.reindex(submission[ID_COL]).to_numpy())
pred_index = np.searchsorted(classes, submission[SUBMIT_COL].to_numpy())
print(f"{METRIC_NAME} di test (kunci): {score(truth_index, pred_index):.4f}")
print(f"{METRIC_NAME} OOF train: {oof_score:.4f}")